In [1]:
import numpy as np
import pandas as pd
import datetime as dt
from jqdata import *
import statsmodels.api as sm

In [2]:
# 信息设置
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
#
# 基金市场研究
#

In [7]:
#  参数
index = '399311.XSHE'

In [8]:
# 当前
dt_now = dt.datetime.now().date()
dt_now

datetime.date(2023, 4, 25)

In [9]:
# 股票池
stocks = get_index_stocks(index)

In [11]:
# 数据段
dt_end = dt.date(2023, 3, 31)
reportId = 403001 # 年度403006，半年度403005，第一季度403001

In [12]:
# 测试
# 注意：code和symbol的格式，没有后缀！
stock = '600519'
df = finance.run_query(query(
        finance.FUND_PORTFOLIO_STOCK
    ).filter(
        finance.FUND_PORTFOLIO_STOCK.symbol == stock,
        finance.FUND_PORTFOLIO_STOCK.period_end == dt_end,
        finance.FUND_PORTFOLIO_STOCK.report_type_id == reportId,
    ).limit(5)
    )
df

,id,code,period_start,period_end,pub_date,report_type_id,report_type,rank,symbol,name,shares,market_cap,proportion
0,150658167,000165,2023-01-01,2023-03-31,2023-04-20,403001,第一季度,3,600519,贵州茅台,9701.0,17655820.0,2.38
1,150658186,000259,2023-01-01,2023-03-31,2023-04-20,403001,第一季度,2,600519,贵州茅台,7500.0,13650000.0,2.79
2,150658196,000336,2023-01-01,2023-03-31,2023-04-20,403001,第一季度,2,600519,贵州茅台,108600.0,197652000.0,5.26
3,150658226,000551,2023-01-01,2023-03-31,2023-04-20,403001,第一季度,2,600519,贵州茅台,30000.0,54600000.0,5.79
4,150658275,000845,2023-01-01,2023-03-31,2023-04-20,403001,第一季度,1,600519,贵州茅台,2200.0,4004000.0,8.17


In [13]:
# 逐股提取基金持仓
fportfolio = pd.Series()
for s in stocks:
    stock = s[:6]
    df = finance.run_query(query(
            finance.FUND_PORTFOLIO_STOCK
        ).filter(
            finance.FUND_PORTFOLIO_STOCK.symbol == stock,
            finance.FUND_PORTFOLIO_STOCK.period_end == dt_end,
            finance.FUND_PORTFOLIO_STOCK.report_type_id == reportId,
        ).order_by(finance.FUND_PORTFOLIO_STOCK.market_cap.desc())                           
        )
    total_value = 1e-8*df.market_cap.sum()
    fportfolio[s] = total_value
    print(s, get_security_info(s).display_name, total_value) # 提取数据过程很长，需要不停输出信息以示运行进度

000001.XSHE 平安银行 98.39244423259998
000002.XSHE 万科A 91.6966556208
000009.XSHE 中国宝安 0.11706552
000012.XSHE 南玻A 0.6050407
000021.XSHE 深科技 2.36449224
000027.XSHE 深圳能源 0.01877603
000031.XSHE 大悦城 0.00860884
000032.XSHE 深桑达A 39.5905570206
000035.XSHE 中国天楹 6.3627285405
000039.XSHE 中集集团 0.05161863
000050.XSHE 深天马A 0.05961272
000060.XSHE 中金岭南 2.04082584
000062.XSHE 深圳华强 0.015696881200000002
000063.XSHE 中兴通讯 159.036293548
000066.XSHE 中国长城 7.772955081999999
000069.XSHE 华侨城A 12.400369086
000089.XSHE 深圳机场 0.034263
000100.XSHE TCL科技 20.222813954499998
000155.XSHE 川能动力 2.1710451056
000156.XSHE 华数传媒 0.19535688
000157.XSHE 中联重科 0.57488454
000166.XSHE 申万宏源 24.7248515283
000301.XSHE 东方盛虹 12.17717211
000333.XSHE 美的集团 215.10357712130002
000338.XSHE 潍柴动力 19.5455300118
000400.XSHE 许继电气 8.05026924
000401.XSHE 冀东水泥 2.1310582216
000402.XSHE 金融街 0.08352276
000403.XSHE 派林生物 2.6934321088
000408.XSHE 藏格矿业 1.9128034928
000415.XSHE 渤海租赁 0.0216426
000423.XSHE 东阿阿胶 44.2425453946
000425.XSHE 徐工机械 23.9731442181
000513.XSH

002600.XSHE 领益智造 2.1054506658
002601.XSHE 龙佰集团 22.502224597599998
002602.XSHE 世纪华通 6.298716849600001
002603.XSHE 以岭药业 7.6757664273
002607.XSHE 中公教育 5.61704074
002608.XSHE 江苏国信 0.0
002610.XSHE 爱康科技 0.0
002617.XSHE 露笑科技 0.2764003872
002624.XSHE 完美世界 6.8217184604
002625.XSHE 光启技术 10.2302599392
002626.XSHE 金达威 0.6232285211999999
002643.XSHE 万润股份 20.7540412174
002648.XSHE 卫星化学 43.31489184
002653.XSHE 海思科 2.3819072163999997
002670.XSHE 国盛金控 0.0
002673.XSHE 西部证券 0.0
002683.XSHE 广东宏大 2.208829767
002690.XSHE 美亚光电 1.200290287
002701.XSHE 奥瑞金 0.9290533428000001
002705.XSHE 新宝股份 0.32171520000000003
002706.XSHE 良信股份 2.764903545
002709.XSHE 天赐材料 121.8744971412
002714.XSHE 牧原股份 125.04988258
002727.XSHE 一心堂 66.4035509184
002736.XSHE 国信证券 0.3159200444
002738.XSHE 中矿资源 45.69242139600001
002739.XSHE 万达电影 24.715204085
002745.XSHE 木林森 0.0
002747.XSHE 埃斯顿 10.146812090800001
002756.XSHE 永兴材料 56.742943443
002773.XSHE 康弘药业 1.0932424
002791.XSHE 坚朗五金 11.883651574300002
002797.XSHE 第一创业 0.0
002812.XSHE 恩捷股份 151.

600271.XSHG 航天信息 2.1256153816
600273.XSHG 嘉化能源 0.0
600276.XSHG 恒瑞医药 322.3741689904
600282.XSHG 南钢股份 10.4173648787
600295.XSHG 鄂尔多斯 0.0264941
600297.XSHG 广汇汽车 0.1020978
600298.XSHG 安琪酵母 22.6900315675
600299.XSHG 安迪苏 0.0
600309.XSHG 万华化学 303.7419395448
600315.XSHG 上海家化 4.9815486
600316.XSHG 洪都航空 7.214574369599999
600323.XSHG 瀚蓝环境 8.6198572845
600325.XSHG 华发股份 47.1602763016
600329.XSHG 达仁堂 14.7010011414
600330.XSHG 天通股份 1.4642474568000001
600332.XSHG 白云山 5.5317404788
600338.XSHG 西藏珠峰 13.508088879999997
600339.XSHG 中油工程 0.0
600340.XSHG 华夏幸福 0.0
600346.XSHG 恒力石化 22.729830552000003
600348.XSHG 华阳股份 11.449216314000001
600350.XSHG 山东高速 6.1182100836
600352.XSHG 浙江龙盛 5.800200942900001
600362.XSHG 江西铜业 0.1668734359
600369.XSHG 西南证券 0.0
600372.XSHG 中航电子 61.1463526224
600373.XSHG 中文传媒 3.8306957919999998
600376.XSHG 首开股份 0.40129824000000003
600377.XSHG 宁沪高速 0.40866336200000003
600378.XSHG 昊华科技 43.3514721795
600380.XSHG 健康元 4.37707584
600383.XSHG 金地集团 28.014744072
600390.XSHG 五矿资本 0.11902891
600392.X

601669.XSHG 中国电建 14.800875901
601677.XSHG 明泰铝业 21.984599286799998
601688.XSHG 华泰证券 69.0371580375
601689.XSHG 拓普集团 86.74634884720001
601696.XSHG 中银证券 0.00335088
601698.XSHG 中国卫通 0.8953855731999998
601699.XSHG 潞安环能 54.537942772600005
601717.XSHG 郑煤机 5.6630499628
601718.XSHG 际华集团 0.0
601727.XSHG 上海电气 2.1842320188
601728.XSHG 中国电信 73.3757961102
601766.XSHG 中国中车 16.0060560768
601777.XSHG 力帆科技 0.0
601778.XSHG 晶科科技 0.4540856589999999
601788.XSHG 光大证券 2.5059531288
601799.XSHG 星宇股份 38.077864780999995
601800.XSHG 中国交建 16.489981622400002
601808.XSHG 中海油服 2.2792185000000003
601816.XSHG 京沪高铁 23.852505666600003
601818.XSHG 光大银行 6.698329386
601825.XSHG 沪农商行 0.79377408
601828.XSHG 美凯龙 0.01113024
601838.XSHG 成都银行 72.5063422884
601857.XSHG 中国石油 18.3906699136
601865.XSHG 福莱特 57.1642440499
601866.XSHG 中远海发 0.0
601868.XSHG 中国能建 5.1756242031
601869.XSHG 长飞光纤 0.44298
601872.XSHG 招商轮船 39.4576651521
601877.XSHG 正泰电器 7.578260885899999
601878.XSHG 浙商证券 1.4877898339000002
601880.XSHG 辽港股份 0.0
601881.XSHG 中国银河 2.1

In [14]:
fportfolio.head()

000001.XSHE    98.392444
000002.XSHE    91.696656
000009.XSHE     0.117066
000012.XSHE     0.605041
000021.XSHE     2.364492
dtype: float64

In [15]:
fweight = 100 * fportfolio / fportfolio.sum()
fweight = fweight.sort_values(ascending=False)
fweight.head()

600519.XSHG    5.544621
300750.XSHE    4.267183
000858.XSHE    2.919360
000568.XSHE    2.805544
603259.XSHG    1.542663
dtype: float64

In [16]:
index_weight = get_index_weights(index, dt_end)
index_weight = index_weight.sort_values(by='weight', ascending=False)
index_weight[:10]

,date,weight,display_name
code,,,
600519.XSHG,2023-03-31,3.8417,贵州茅台
300750.XSHE,2023-03-31,1.9123,宁德时代
601318.XSHG,2023-03-31,1.6312,中国平安
600036.XSHG,2023-03-31,1.5410,招商银行
000858.XSHE,2023-03-31,1.3252,五粮液
601166.XSHG,2023-03-31,1.0182,兴业银行
000333.XSHE,2023-03-31,0.9669,美的集团
601012.XSHG,2023-03-31,0.8570,隆基绿能
002594.XSHE,2023-03-31,0.8077,比亚迪


In [17]:
fdelta = pd.Series()
for s in index_weight.index:
    if s in fweight.index:
        fdelta[s] = fweight[s] - index_weight.weight[s]
fdelta = fdelta.sort_values(ascending=False)

In [18]:
# 最受欢迎的个股
the_best_delta = fdelta[fdelta > 0].sort_values(ascending=False)
the_best_list = the_best_delta.index.tolist()
the_best = index_weight.loc[the_best_list]
the_best['delta'] = the_best_delta
print(len(the_best))
the_best.head(15)

262


,date,weight,display_name,delta
code,,,,
300750.XSHE,2023-03-31,1.9123,宁德时代,2.354883
000568.XSHE,2023-03-31,0.7014,泸州老窖,2.104144
600519.XSHG,2023-03-31,3.8417,贵州茅台,1.702921
000858.XSHE,2023-03-31,1.3252,五粮液,1.594160
600809.XSHG,2023-03-31,0.3993,山西汾酒,1.074884
603259.XSHG,2023-03-31,0.5668,药明康德,0.975863
300760.XSHE,2023-03-31,0.5316,迈瑞医疗,0.881247
300015.XSHE,2023-03-31,0.3438,爱尔眼科,0.843466
300274.XSHE,2023-03-31,0.3868,阳光电源,0.819719


In [19]:
# 最被抛弃的个股
the_worst_delta = fdelta[fdelta < 0].sort_values(ascending=True)
the_worst_list = the_worst_delta.index.tolist()
the_worst = index_weight.loc[the_worst_list]
the_worst['delta'] = the_worst_delta
print(len(the_worst))
the_worst.head(15)

738


,date,weight,display_name,delta
code,,,,
601318.XSHG,2023-03-31,1.6312,中国平安,-0.863655
601166.XSHG,2023-03-31,1.0182,兴业银行,-0.653591
600036.XSHG,2023-03-31,1.5410,招商银行,-0.597618
600900.XSHG,2023-03-31,0.7767,长江电力,-0.443900
000333.XSHE,2023-03-31,0.9669,美的集团,-0.432665
601398.XSHG,2023-03-31,0.5801,工商银行,-0.426438
000725.XSHE,2023-03-31,0.5221,京东方A,-0.415961
601328.XSHG,2023-03-31,0.4503,交通银行,-0.366690
601816.XSHG,2023-03-31,0.4045,京沪高铁,-0.345260
